In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv('C:\\Users\\Abdullah\\Desktop\\Code\\Sentiment Based Stock\\Sentiment Based Stocks\\results\\final_dataset_labeled.csv')
df['hour'] = pd.to_datetime(df['hour'])

price_cols     = ['open', 'high', 'low', 'close', 'volume',
                  'returns', 'volatility', 'rsi', 'ma_7', 'ma_30']
sentiment_cols = ['mean_finbert', 'mean_bullish', 'mean_bearish',
                  'mean_strength', 'post_count']

In [3]:

TARGET = 'label_3h'

train = df[df['hour'] < '2025-07-01']
test  = df[df['hour'] >= '2025-07-01']

print("Train size:", len(train))
print("Test size:", len(test))

Train size: 2146
Test size: 1345


In [4]:
price_scaler    = StandardScaler()
sentiment_scaler = StandardScaler()

train_price = price_scaler.fit_transform(train[price_cols])
test_price  = price_scaler.transform(test[price_cols])

train_sent  = sentiment_scaler.fit_transform(train[sentiment_cols])
test_sent   = sentiment_scaler.transform(test[sentiment_cols])

train_regime = train['regime'].values
test_regime  = test['regime'].values

train_y = train[TARGET].values
test_y  = test[TARGET].values

SEQ_LEN = 24

In [5]:
class RegimeDataset(Dataset):
    def __init__(self, price, sent, regime, y):
        self.price  = []
        self.sent   = []
        self.regime = []
        self.y      = []
        for i in range(SEQ_LEN, len(price)):
            self.price.append(price[i-SEQ_LEN:i])
            self.sent.append(sent[i-SEQ_LEN:i])
            self.regime.append(regime[i])
            self.y.append(y[i])
        self.price  = torch.tensor(np.array(self.price),  dtype=torch.float32)
        self.sent   = torch.tensor(np.array(self.sent),   dtype=torch.float32)
        self.regime = torch.tensor(np.array(self.regime), dtype=torch.float32)
        self.y      = torch.tensor(np.array(self.y),      dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.price[idx], self.sent[idx], self.regime[idx], self.y[idx]

train_ds     = RegimeDataset(train_price, train_sent, train_regime, train_y)
test_ds      = RegimeDataset(test_price,  test_sent,  test_regime,  test_y)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)


In [6]:
class RegimeAwareFusion(nn.Module):
    def __init__(self, price_input, sent_input, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()

        # Branch 1: Price BiLSTM
        self.price_lstm = nn.LSTM(price_input, hidden_size,
                                  num_layers=num_layers,
                                  batch_first=True,
                                  bidirectional=True,
                                  dropout=dropout)
        self.price_fc = nn.Linear(hidden_size * 2, 32)

        self.sent_lstm = nn.LSTM(sent_input, hidden_size,
                                 num_layers=num_layers,
                                 batch_first=True,
                                 bidirectional=True,
                                 dropout=dropout)
        self.sent_fc = nn.Linear(hidden_size * 2, 32)

        self.regime_gate = nn.Linear(1, 1)

        self.classifier = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, price, sent, regime):
        p_out, _ = self.price_lstm(price)
        p_emb    = torch.relu(self.price_fc(p_out[:, -1, :]))

        s_out, _ = self.sent_lstm(sent)
        s_emb    = torch.relu(self.sent_fc(s_out[:, -1, :]))

        regime   = regime.unsqueeze(1)
        w        = torch.sigmoid(self.regime_gate(regime))

        fused    = w * s_emb + (1 - w) * p_emb

        return self.classifier(fused).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

model     = RegimeAwareFusion(
    price_input=len(price_cols),
    sent_input=len(sentiment_cols)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
EPOCHS    = 30

Device: cuda


In [7]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for p_batch, s_batch, r_batch, y_batch in train_loader:
        p_batch = p_batch.to(device)
        s_batch = s_batch.to(device)
        r_batch = r_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(p_batch, s_batch, r_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader):.4f}")

Epoch 5/30 — Loss: 0.6908
Epoch 10/30 — Loss: 0.6875
Epoch 15/30 — Loss: 0.6788
Epoch 20/30 — Loss: 0.6615
Epoch 25/30 — Loss: 0.6404
Epoch 30/30 — Loss: 0.5967


In [8]:
model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for p_batch, s_batch, r_batch, y_batch in test_loader:
        p_batch = p_batch.to(device)
        s_batch = s_batch.to(device)
        r_batch = r_batch.to(device)
        probs   = model(p_batch, s_batch, r_batch).cpu().numpy()
        probs   = np.atleast_1d(probs)
        preds   = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

In [9]:
print("\n── Regime-Aware Fusion Results (3h) ──")
print(f"Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
print(f"F1        : {f1_score(all_labels, all_preds):.4f}")
print(f"AUC       : {roc_auc_score(all_labels, all_probs):.4f}")


── Regime-Aware Fusion Results (3h) ──
Accuracy  : 0.5117
Precision : 0.5200
Recall    : 0.5778
F1        : 0.5474
AUC       : 0.5084


In [10]:
TARGET = 'label_6h'

train = df[df['hour'] < '2025-07-01']
test  = df[df['hour'] >= '2025-07-01']

print("Train size:", len(train))
print("Test size:", len(test))

Train size: 2146
Test size: 1345


In [11]:
price_scaler    = StandardScaler()
sentiment_scaler = StandardScaler()

train_price = price_scaler.fit_transform(train[price_cols])
test_price  = price_scaler.transform(test[price_cols])

train_sent  = sentiment_scaler.fit_transform(train[sentiment_cols])
test_sent   = sentiment_scaler.transform(test[sentiment_cols])

train_regime = train['regime'].values
test_regime  = test['regime'].values

train_y = train[TARGET].values
test_y  = test[TARGET].values

In [12]:

SEQ_LEN = 24

class RegimeDataset(Dataset):
    def __init__(self, price, sent, regime, y):
        self.price  = []
        self.sent   = []
        self.regime = []
        self.y      = []
        for i in range(SEQ_LEN, len(price)):
            self.price.append(price[i-SEQ_LEN:i])
            self.sent.append(sent[i-SEQ_LEN:i])
            self.regime.append(regime[i])
            self.y.append(y[i])
        self.price  = torch.tensor(np.array(self.price),  dtype=torch.float32)
        self.sent   = torch.tensor(np.array(self.sent),   dtype=torch.float32)
        self.regime = torch.tensor(np.array(self.regime), dtype=torch.float32)
        self.y      = torch.tensor(np.array(self.y),      dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.price[idx], self.sent[idx], self.regime[idx], self.y[idx]

train_ds     = RegimeDataset(train_price, train_sent, train_regime, train_y)
test_ds      = RegimeDataset(test_price,  test_sent,  test_regime,  test_y)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)


In [13]:
class RegimeAwareFusion(nn.Module):
    def __init__(self, price_input, sent_input, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()

        self.price_lstm = nn.LSTM(price_input, hidden_size,
                                  num_layers=num_layers,
                                  batch_first=True,
                                  bidirectional=True,
                                  dropout=dropout)
        self.price_fc = nn.Linear(hidden_size * 2, 32)

        self.sent_lstm = nn.LSTM(sent_input, hidden_size,
                                 num_layers=num_layers,
                                 batch_first=True,
                                 bidirectional=True,
                                 dropout=dropout)
        self.sent_fc = nn.Linear(hidden_size * 2, 32)

        self.regime_gate = nn.Linear(1, 1)

        self.classifier = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, price, sent, regime):
        p_out, _ = self.price_lstm(price)
        p_emb    = torch.relu(self.price_fc(p_out[:, -1, :]))

        s_out, _ = self.sent_lstm(sent)
        s_emb    = torch.relu(self.sent_fc(s_out[:, -1, :]))

        regime   = regime.unsqueeze(1)
        w        = torch.sigmoid(self.regime_gate(regime))

        fused    = w * s_emb + (1 - w) * p_emb

        return self.classifier(fused).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

model     = RegimeAwareFusion(
    price_input=len(price_cols),
    sent_input=len(sentiment_cols)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
EPOCHS    = 30

Device: cuda


In [14]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for p_batch, s_batch, r_batch, y_batch in train_loader:
        p_batch = p_batch.to(device)
        s_batch = s_batch.to(device)
        r_batch = r_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(p_batch, s_batch, r_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader):.4f}")

Epoch 5/30 — Loss: 0.6906
Epoch 10/30 — Loss: 0.6820
Epoch 15/30 — Loss: 0.6531
Epoch 20/30 — Loss: 0.6174
Epoch 25/30 — Loss: 0.5377
Epoch 30/30 — Loss: 0.4854


In [15]:
model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for p_batch, s_batch, r_batch, y_batch in test_loader:
        p_batch = p_batch.to(device)
        s_batch = s_batch.to(device)
        r_batch = r_batch.to(device)
        probs   = model(p_batch, s_batch, r_batch).cpu().numpy()
        probs   = np.atleast_1d(probs)
        preds   = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

In [16]:
print("\n── Regime-Aware Fusion Results (6h) ──")
print(f"Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
print(f"F1        : {f1_score(all_labels, all_preds):.4f}")
print(f"AUC       : {roc_auc_score(all_labels, all_probs):.4f}")


── Regime-Aware Fusion Results (6h) ──
Accuracy  : 0.4837
Precision : 0.5036
Recall    : 0.6090
F1        : 0.5513
AUC       : 0.4902
